# Molecular Ground State via VQE — H₂ Potential Energy Surface

This notebook demonstrates how to compute the **potential energy surface (PES)** of the hydrogen molecule (H₂) using Divi's VQE framework.

We'll use two key Divi features:
- **`VQEHyperparameterSweep`** — sweeps over ansätze × Hamiltonians in parallel
- **`VQE`** — finds the ground-state energy for each configuration

---

## Step 1 — Imports

In [ ]:
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

from divi.backends import ParallelSimulator
from divi.qprog import (
    VQE,
    UCCSDAnsatz,
    GenericLayerAnsatz,
    VQEHyperparameterSweep,
)
from divi.qprog.optimizers import MonteCarloOptimizer

%matplotlib inline

## Step 2 — Build molecular Hamiltonians

PennyLane's quantum chemistry module computes the electronic Hamiltonian (STO-3G basis, Jordan-Wigner mapping) for H₂ at each bond length, yielding a 4-qubit operator.

In [ ]:
BOND_LENGTHS = np.linspace(0.3, 2.5, 10).tolist()

hamiltonians = {}
for r_angstrom in BOND_LENGTHS:
    r_bohr = r_angstrom / 0.529177  # Å → Bohr
    half = r_bohr / 2.0
    H, n_qubits = qml.qchem.molecular_hamiltonian(
        symbols=["H", "H"],
        coordinates=np.array([0.0, 0.0, -half, 0.0, 0.0, half]),
    )
    hamiltonians[r_angstrom] = H

print(f"Built {len(hamiltonians)} Hamiltonians ({n_qubits} qubits each)")
print(f"Bond lengths: {min(BOND_LENGTHS):.2f} – {max(BOND_LENGTHS):.2f} Å")

## Step 3 — Choose ansätze

We compare two ansätze:
- **UCCSD** — the chemistry gold standard (unitary coupled cluster singles & doubles)
- **GenericLayer (RY+RZ / CNOT)** — a hardware-efficient-style ansatz with fewer gates

In [ ]:
ansatze = [
    UCCSDAnsatz(),
    GenericLayerAnsatz(
        gate_sequence=[qml.RY, qml.RZ],
        entangler=qml.CNOT,
        entangling_layout="linear",
    ),
]

print(f"Ansätze: {[a.name for a in ansatze]}")

## Step 4 — Run the VQE sweep

Divi's `VQEHyperparameterSweep` creates one VQE instance per (ansatz, Hamiltonian) combination and runs them all via `ProgramBatch`.

⏱️ **This step takes a couple of minutes** — it runs 20 independent VQE optimizations (2 ansätze × 10 bond lengths).

In [ ]:
backend = ParallelSimulator(shots=2_000, n_processes=4)

sweep = VQEHyperparameterSweep(
    ansatze=ansatze,
    hamiltonians=hamiltonians,
    optimizer=MonteCarloOptimizer(population_size=10, n_best_sets=3),
    max_iterations=10,
    backend=backend,
    n_electrons=2,
)

print("Creating VQE programs...")
sweep.create_programs()
print(f"  {len(sweep.programs)} VQE instances created")

print("\n🚀 Running sweep...")
sweep.run(blocking=True)
print("  Sweep complete!")

## Step 5 — Extract and display results

In [ ]:
# Aggregate results
best_config, best_energy = sweep.aggregate_results()

# Extract PES data
pes_data = {}
for (ansatz_name, bond_length), program in sweep.programs.items():
    energy = program.best_loss

    if ansatz_name not in pes_data:
        pes_data[ansatz_name] = {"bond_lengths": [], "energies": []}

    pes_data[ansatz_name]["bond_lengths"].append(bond_length)
    pes_data[ansatz_name]["energies"].append(energy)

# Sort by bond length
for ansatz_name in pes_data:
    pairs = sorted(zip(
        pes_data[ansatz_name]["bond_lengths"],
        pes_data[ansatz_name]["energies"],
    ))
    pes_data[ansatz_name]["bond_lengths"] = [p[0] for p in pairs]
    pes_data[ansatz_name]["energies"] = [p[1] for p in pairs]

# Print summary
for ansatz_name, data in pes_data.items():
    min_idx = int(np.argmin(data["energies"]))
    print(f"{ansatz_name}:")
    print(f"  Equilibrium ≈ {data['bond_lengths'][min_idx]:.3f} Å")
    print(f"  Energy      = {data['energies'][min_idx]:.6f} Ha")

print(f"\n🏆 Best: {best_config[0]} at r={best_config[1]:.2f} Å → E = {best_energy:.6f} Ha")

## Step 6 — Plot the potential energy surface

The classic Morse-like curve: energy dips at the equilibrium bond length and rises at both short (repulsive) and long (dissociation) distances.

In [ ]:
COLORS = {
    "UCCSDAnsatz": "#7c4dff",
    "GenericLayerAnsatz": "#00bcd4",
}

fig, ax = plt.subplots(figsize=(10, 6))

for ansatz_name, data in pes_data.items():
    color = COLORS.get(ansatz_name, "#333333")
    ax.plot(
        data["bond_lengths"], data["energies"],
        "-o", color=color, linewidth=2, markersize=6,
        label=ansatz_name, alpha=0.9,
    )
    # Mark equilibrium
    min_idx = int(np.argmin(data["energies"]))
    ax.plot(
        data["bond_lengths"][min_idx], data["energies"][min_idx],
        "*", color=color, markersize=18, zorder=5,
    )
    ax.annotate(
        f"  {data['energies'][min_idx]:.4f} Ha",
        (data["bond_lengths"][min_idx], data["energies"][min_idx]),
        color=color, fontsize=10, fontweight="bold",
    )

ax.set_xlabel("Bond Length (Å)", fontsize=13)
ax.set_ylabel("Ground-State Energy (Hartree)", fontsize=13)
ax.set_title("H₂ Potential Energy Surface — VQE with Divi", fontsize=15, fontweight="bold", pad=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()